# Exercise 3 - indices

## Imports

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import spectral.io.envi as envi
from pystac_client import Client
import planetary_computer
from datetime import datetime

ACQUISITION_DATE = '2025-06-17'

BASE_DIR = Path().cwd() / "lab_5" / "data" / "you-shall-not-pass" / "Obrazy lotnicze"
hdr_path = BASE_DIR / '221000_Odra_HS_Blok_A_008_VS_join_atm.hdr'

print(f"Target Date: {ACQUISITION_DATE}")
print(f"Airborne Header Path: {hdr_path}")

## Load Airborne Data Cube

Only a subset is loaded to save memory ;D

In [ ]:
if not hdr_path.exists():
    raise FileNotFoundError(f"Cannot find airborne data at {hdr_path}")

img = envi.open(hdr_path)
meta = img.metadata
wavelengths = np.array([float(x) for x in meta['wavelength']])

# Read a subset (1000x1000) from the center to keep memory usage low
cy, cx = img.nrows // 2, img.ncols // 2
cube = img.read_subregion((cy-500, cy+500), (cx-500, cx+500))
print(f"Loaded subset shape: {cube.shape}")

## Display False-Color Composite

Appropriate bands for R, G and B are selected to represent RGB visualization for the airborn data

In [ ]:
r_idx = np.argmin(np.abs(wavelengths - 670))
g_idx = np.argmin(np.abs(wavelengths - 560))
b_idx = np.argmin(np.abs(wavelengths - 490))

rgb_air = cube[:, :, [r_idx, g_idx, b_idx]]
# Simple stretch for visualization
rgb_air = (rgb_air - np.percentile(rgb_air, 2)) / (np.percentile(rgb_air, 98) - np.percentile(rgb_air, 2))
rgb_air = np.clip(rgb_air, 0, 1)

plt.figure(figsize=(8, 8))
plt.imshow(rgb_air)
plt.title("Airborne False Color RGB")
plt.axis('off')
plt.show()

## Calculate Water Quality Indices for the airborn data

Formulas used:
- **Chl-a**: Ratio of 705nm / 665nm (Red edge / Red)
- **DOC**: Ratio of 560nm / 665nm (Green / Red)
- **Turbidity**: Reflected intensity at 700nm.

### Explanation of the index exraction method:

Since the bands might not be precisely at a given wavelength, the value is interpolated from the nearest bands. It removes wanted wavelength from the spectrum and then selects position where the difference is minimal (it's absolute at this point).

In [ ]:
def safe_ratio(n, d, eps=1e-6):
    return n / (d + eps)

idx_705 = np.argmin(np.abs(wavelengths - 705))
idx_665 = np.argmin(np.abs(wavelengths - 665))
idx_560 = np.argmin(np.abs(wavelengths - 560))
idx_700 = np.argmin(np.abs(wavelengths - 700))

chla_air = safe_ratio(cube[:, :, idx_705], cube[:, :, idx_665])
doc_air = safe_ratio(cube[:, :, idx_560], cube[:, :, idx_665])
turb_air = cube[:, :, idx_700]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (data, name) in enumerate(zip([chla_air, doc_air, turb_air], ['Chl-a', 'DOC', 'Turbidity'])):
    im = axes[i].imshow(data, cmap='viridis')
    axes[i].set_title(f"Airborne {name}")
    plt.colorbar(im, ax=axes[i])
plt.show()

## Download Sentinel-2 Data for a given acquisition date


In [ ]:
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=planetary_computer.sign_inplace)

# AOI: Odra river coordinates near the airborne flight
bbox = [14.4, 52.8, 14.8, 53.2]

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=ACQUISITION_DATE,
    query={"eo:cloud_cover": {"lt": 20}}
)

items = list(search.items())
if not items:
    print("No Sentinel-2 scenes found for the specific date. Checking nearest...")
    search = catalog.search(collections=["sentinel-2-l2a"], bbox=bbox, datetime="2025-06-10/2025-06-25", sortby="datetime")
    items = list(search.items())

best_item = items[0]
print(f"Selected Sentinel-2 Scene: {best_item.id} ({best_item.datetime})")

# URLs of bands that will be analysed, as it was done in previous labs
assets = best_item.assets
bands_s2 = {
    'B03': assets['B03'].href, # Green
    'B04': assets['B04'].href, # Red
    'B05': assets['B05'].href, # Red Edge 1 (~705nm)
    'B11': assets['B11'].href  # SWIR 1 (used for turbidity proxy)
}

## Calculate Water Quality Indices for Sentinel-2 data



In [ ]:
data_s2 = {}
with rasterio.open(bands_s2['B04']) as src:
    # Read a 500x500 window corresponding to the general area
    win = rasterio.windows.Window(1000, 1000, 500, 500)
    data_s2['B04'] = src.read(1, window=win).astype(float) / 10000.0

with rasterio.open(bands_s2['B03']) as src:
    data_s2['B03'] = src.read(1, window=win).astype(float) / 10000.0

with rasterio.open(bands_s2['B05']) as src:
    # B05 is 20m resolution, we upsample to 10m to match B04
    data_s2['B05'] = src.read(1, out_shape=data_s2['B04'].shape, resampling=rasterio.enums.Resampling.bilinear).astype(float) / 10000.0

chla_s2 = safe_ratio(data_s2['B05'], data_s2['B04'])
doc_s2 = safe_ratio(data_s2['B03'], data_s2['B04'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(chla_s2, cmap='viridis', vmin=0, vmax=2)
axes[0].set_title("Sentinel-2 Chl-a Proxy")
axes[1].imshow(doc_s2, cmap='viridis', vmin=0, vmax=2)
axes[1].set_title("Sentinel-2 DOC Proxy")
plt.show()

## Compare Results (values)

Mean of the values is compared, the difference is significant, as the result of the resolution difference (1m vs 10m).

In [ ]:
print(f"Airborne Chl-a: {np.nanmean(chla_air):.4f}")
print(f"Sentinel-2 Chl-a: {np.nanmean(chla_s2):.4f}")
print(f"Relative Difference: {abs(np.nanmean(chla_air)-np.nanmean(chla_s2))/np.nanmean(chla_air)*100:.2f}%")

print(f"\nAirborne DOC: {np.nanmean(doc_air):.4f}")
print(f"Sentinel-2 DOC: {np.nanmean(doc_s2):.4f}")
print(f"Relative Difference: {abs(np.nanmean(doc_air)-np.nanmean(doc_s2))/np.nanmean(doc_air)*100:.2f}%")